# VoiceDiary AI — Live Cloud GPU Platform
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

> **Quick Start:** Click **Runtime → Run all** (`Ctrl+F9`) · Real-time streaming lecture notes with inline speaker renaming cards!

In [ ]:
!pip install -q --no-cache-dir faster-whisper speechbrain gradio soundfile torchaudio


In [ ]:

import os, time, tempfile, json, re, urllib.request, gc
import numpy as np
import soundfile as sf
import gradio as gr
import torch
import torchaudio
from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

# ─── Hardware Acceleration ───
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (AVX2)"
device_type = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = "float16" if device_type == "cuda" else "int8"
print(f"⚡ Hardware: {gpu_name} | Compute: {compute_dtype}")

# ─── Model Hub Cache ───
_model_cache = {}
def get_model(name):
    if name not in _model_cache:
        print(f"Loading Whisper model: {name}…")
        os.makedirs("/content/models/whisper", exist_ok=True)
        _model_cache[name] = WhisperModel(name, device=device_type, compute_type=compute_dtype,
            num_workers=2, download_root="/content/models/whisper")
    return _model_cache[name]

print("Pre-warming Large-v3-Turbo default model…")
get_model("large-v3-turbo")

print("Pre-warming SpeechBrain ECAPA-TDNN neural diarizer…")
os.makedirs("/content/models/ecapa", exist_ok=True)
embedder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/models/ecapa", run_opts={"device": device_type})
print("✓ VoiceDiary AI Engine ready.")

# ─── 1. Desktop Urdu Normalizer ───
CHAR_MAPPINGS = {
    '\u0643': '\u06a9', '\u064a': '\u06cc', '\u0649': '\u06cc', '\u06c2': '\u06c1',
    '\u0647': '\u06c1', '\u06c3': '\u06c1', '\u0629': '\u06c1', '\u0624': '\u0648',
}
HAMZA_CORRECTIONS = {
    'آو': 'آؤ', 'جاو': 'جاؤ', 'کھاو': 'کھاؤ', 'پیو': 'پیئو', 'سناو': 'سناؤ', 'بتاو': 'بتاؤ',
    'دکھاو': 'دکھاؤ', 'گاو': 'گاؤ', 'لاو': 'لاؤ', 'آۓ': 'آئے', 'گۓ': 'گئے', 'ہوۓ': 'ہوئے',
    'گاۓ': 'گائے', 'بجاۓ': 'بجائے', 'جاۓ': 'جائے', 'پاۓ': 'پائے', 'کھاۓ': 'کھائے',
    'سناۓ': 'سنائے', 'بتاۓ': 'بتائے', 'دکھاۓ': 'دکھائے', 'چاہیۓ': 'چاہئے', 'کیجئے': 'کیجیئے',
    'دیجئے': 'دیجیئے', 'لیجئے': 'لیجیئے'
}
ORTHOGRAPHIC_CORRECTIONS = {
    'توتہ': 'توتا', 'گانہ': 'گانا', 'طوطا': 'توتا', 'بلکل': 'بالکل',
    'انشااللہ': 'ان شاء اللہ', 'ماشااللہ': 'ما شاء اللہ', 'صحیح': 'صحیح'
}

class UrduNormalizer:
    @staticmethod
    def normalize(text: str) -> str:
        if not text: return ""
        text = re.sub(r'[\u200b-\u200f\ufeff]', '', text)
        chars = [CHAR_MAPPINGS.get(c, c) for c in text]
        text = "".join(chars)
        for wrong, right in HAMZA_CORRECTIONS.items():
            text = re.sub(r'\b' + re.escape(wrong) + r'\b', right, text)
        for wrong, right in ORTHOGRAPHIC_CORRECTIONS.items():
            text = re.sub(r'\b' + re.escape(wrong) + r'\b', right, text)
        text = re.sub(r'\s+([۔،؟!])', r'\1', text)
        text = re.sub(r'([.?!,])\1+', r'\1', text)
        return re.sub(r'\s+', ' ', text).strip()

# ─── 2. Desktop English Post-Processor ───
ENGLISH_CONFUSION_PAIRS = [
    (r'\bavailability\s+to\s+convey\b', 'the ability to convey'),
    (r'\bavailability\s+to\b', 'ability to'),
    (r'\bfocus\s+and\s+availability\b', 'focus, and the ability'),
    (r'\bfocus\s+and\s+the\s+availability\b', 'focus, and the ability'),
    (r'\broles\s+and\s+the\s+world\b', 'role in the world'),
    (r'\brole\s+and\s+the\s+world\b', 'role in the world'),
    (r'\btheir\s+roles\s+and\s+the\s+world\b', 'their role in the world'),
    (r'\bentertains\s+inform\b', 'entertain, inform'),
    (r'\bentertains,\s*inform\b', 'entertain, inform'),
    (r'\bcontent\s+that\s+entertains\s+and\s+inform\b', 'content that entertains, informs'),
]

def post_process_english(text: str) -> str:
    if not text: return ""
    processed = text
    for pattern, replacement in ENGLISH_CONFUSION_PAIRS:
        processed = re.sub(pattern, replacement, processed, flags=re.IGNORECASE)
    processed = re.sub(r'\s+([,.:;?!])', r'\1', processed)
    processed = re.sub(r'([.?!,])\1+', r'\1', processed)
    processed = re.sub(r'\s+', ' ', processed).strip()
    if processed:
        processed = processed[0].upper() + processed[1:]
    return processed

# ─── 3. Desktop Romanizer ───
URDU_WORD_DICT = {
    'میں': 'mein', 'ہم': 'hum', 'تم': 'tum', 'آپ': 'aap', 'تو': 'tu', 'یہ': 'yeh', 'وہ': 'woh',
    'اس': 'is', 'ان': 'un', 'انھیں': 'unhein', 'انہیں': 'unhein', 'ہمیں': 'humein', 'تمہیں': 'tumhein',
    'مجھے': 'mujhe', 'اسے': 'use', 'کا': 'ka', 'کی': 'ki', 'کے': 'ke', 'کو': 'ko', 'سے': 'se',
    'پر': 'par', 'تک': 'tak', 'نے': 'ne', 'اور': 'aur', 'یا': 'ya', 'لیکن': 'lekin', 'مگر': 'magar',
    'اگر': 'agar', 'کیونکہ': 'kyunke', 'تو': 'toh', 'بھی': 'bhi', 'ہی': 'hi', 'ساتھ': 'saath',
    'کیا': 'kya', 'کون': 'kaun', 'کب': 'kab', 'کہاں': 'kahan', 'کیسے': 'kaise', 'کیوں': 'kyun',
    'کتنا': 'kitna', 'کس': 'kis', 'ہے': 'hai', 'ہیں': 'hain', 'ہو': 'ho', 'ہوں': 'hoon',
    'تھا': 'tha', 'تھی': 'thi', 'تھے': 'thay', 'کر': 'kar', 'کرنا': 'karna', 'رہا': 'raha',
    'رہی': 'rahi', 'رہے': 'rahe', 'ہوا': 'hua', 'ہوئی': 'hui', 'ہوئے': 'hue', 'نہیں': 'nahin',
    'صحیح': 'sahi', 'بہت': 'bohot', 'اچھا': 'acha', 'شکریہ': 'shukriya', 'سلام': 'salam',
    'پروجیکٹ': 'project', 'ٹیسٹ': 'test', 'کلاس': 'class', 'سٹوڈنٹ': 'student', 'یونیورسٹی': 'university'
}

def to_roman_urdu(text: str) -> str:
    if not text: return ""
    words = text.split()
    romanized = []
    for w in words:
        clean_w = re.sub(r'[\u064B-\u065F\u0670]', '', w)
        if clean_w in URDU_WORD_DICT:
            romanized.append(URDU_WORD_DICT[clean_w])
        else:
            romanized.append(w)
    return " ".join(romanized)

def sanitize_script(text: str) -> str:
    cleaned = re.sub(r'[\u0900-\u097F\u0F00-\u0FFF\u25A0-\u25FF\uFFFD]+', '', text)
    return cleaned.strip()

# ─── 4. Audio Processing Helpers ───
def load_16k_from_file(path):
    try:
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
        if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        return wav.squeeze().numpy().astype(np.float32)
    except Exception:
        d, sr = sf.read(path)
        if d.ndim > 1: d = d.mean(1)
        d = d.astype(np.float32)
        if sr != 16000:
            n = int(len(d)*16000/sr)
            d = np.interp(np.linspace(0,len(d),n,endpoint=False),np.arange(len(d)),d).astype(np.float32)
        return d

def convert_chunk_to_16k(audio_chunk):
    if audio_chunk is None: return None
    sr, y = audio_chunk
    if y is None or len(y) == 0: return None
    if y.dtype == np.int16:
        y = y.astype(np.float32) / 32768.0
    elif y.dtype == np.int32:
        y = y.astype(np.float32) / 2147483648.0
    elif y.dtype != np.float32:
        y = y.astype(np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != 16000:
        n_samples = int(len(y) * 16000 / sr)
        y = np.interp(np.linspace(0, len(y), n_samples, endpoint=False), np.arange(len(y)), y).astype(np.float32)
    return y

COLORS = ['#6366F1','#10B981','#F59E0B','#EC4899','#06B6D4','#8B5CF6','#F97316','#38BDF8']
MODEL_MAP = {
    'Large-v3-Turbo (809M)': 'large-v3-turbo',
    'Whisper Large-v3 (1.5B)': 'large-v3',
    'Whisper Base (74M)': 'base',
    'Whisper Tiny (39M)': 'tiny',
    'Whisper Small (244M)': 'small',
    'Whisper Medium (769M)': 'medium',
}
LANG_MAP = {
    'Bilingual (Urdu + English)': ('ur', False, False),
    'Pure Urdu Script (اردو)': ('ur', True, False),
    'English Only': ('en', False, True),
    'Roman Urdu (Latin)': ('ur', False, True),
}

PAKISTANI_LECTURE_PROMPT = (
    "Bilingual Pakistani university classroom lecture in mixed English and Urdu. "
    "Discussion on computer science, concepts, code, formulas, assignments, presentations, questions and answers."
)

# ─── Render HTML & Exports from Segments List with Inline Edit Button ───
def render_transcript_ui(segments_list, profiles_dict, spk_names_dict, mkey, elapsed_total):
    if not segments_list:
        empty_t = """<div class='vd-empty'>
          <svg width='54' height='54' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
            <path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/>
          </svg>
          <p>Lecture transcript will stream live here</p>
          <span>Speak into the microphone or upload an audio recording</span>
        </div>"""
        empty_s = """<div class='vd-empty-sm'>
          <svg width='36' height='36' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
            <path d='M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2'/><circle cx='9' cy='7' r='4'/>
            <path d='M23 21v-2a4 4 0 0 0-3-3.87'/><path d='M16 3.13a4 4 0 0 1 0 7.75'/>
          </svg>
          <p>No active speaker profiles</p>
          <span>Start recording to build neural voiceprints</span>
        </div>"""
        empty_e = """<div class='vd-export-row'>
          <span class='vd-dl disabled'>Markdown (.md)</span>
          <span class='vd-dl disabled'>Plain Text (.txt)</span>
          <span class='vd-dl disabled'>Subtitles (.srt)</span>
          <span class='vd-dl disabled'>JSON (.json)</span>
        </div>"""
        return empty_t, empty_s, empty_e, "", None, None, None

    html_parts = []
    plain = []
    srt_parts = []
    json_arr = []
    si = 1

    for seg in segments_list:
        spk = seg["id"]
        display_name = spk_names_dict.get(spk, seg.get("speaker", f"Speaker {spk}"))
        c = COLORS[(spk-1) % len(COLORS)]
        ts = seg["time"]
        txt = seg["text"]
        urdu = any('\u0600' <= ch <= '\u06FF' for ch in txt)
        txt_class = "vd-txt vd-rtl" if urdu else "vd-txt"

        html_parts.append(f"""<div class="vd-seg" style="border-left-color:{c};">
  <div class="vd-seg-body">
    <div class="vd-seg-meta">
      <span class="vd-dot" style="background:{c};box-shadow:0 0 8px {c};"></span>
      <span class="vd-spk-lbl" style="color:{c};">{display_name}</span>
      <span class="vd-time">[{ts}]</span>
    </div>
    <div class="{txt_class}">{txt}</div>
  </div>
</div>""")
        plain.append(f"[{ts}] {display_name}: {txt}")

        def srt_ts(s):
            return f"{int(s//3600):02d}:{int(s%3600//60):02d}:{int(s%60):02d},{int((s-int(s))*1000):03d}"
        srt_parts.append(f"{si}\n{srt_ts(seg['start'])} --> {srt_ts(seg['end'])}\n[{display_name}]: {txt}\n")
        json_arr.append({"speaker": display_name, "id": spk, "start": round(seg['start'],2), "end": round(seg['end'],2), "time": ts, "text": txt})
        si += 1

    # Sidebar with Inline Edit Buttons on each detected speaker card
    sb = []
    for sid, count in profiles_dict.items():
        cc = COLORS[(sid-1) % len(COLORS)]
        sname = spk_names_dict.get(sid, f"Speaker {sid}")
        initial = sname[0].upper() if sname else "S"
        sb.append(f"""<div class="vd-spk-card">
  <div class="vd-spk-av" style="background:{cc};box-shadow:0 0 14px {cc}55">{initial}</div>
  <div class="vd-spk-info">
    <div class="vd-spk-name">{sname}</div>
    <div class="vd-spk-meta">{count} centroid voice print{'s' if count!=1 else ''}</div>
  </div>
  <button class="vd-spk-edit-btn" onclick="renameSpeaker({sid}, '{sname}')" title="Rename {sname}">
    <svg width="13" height="13" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2">
      <path d="M11 4H4a2 2 0 0 0-2 2v14a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2v-7"></path>
      <path d="M18.5 2.5a2.121 2.121 0 0 1 3 3L12 15l-4 1 1-4 9.5-9.5z"></path>
    </svg>
    <span>Edit</span>
  </button>
</div>""")
    if not sb:
        sb = ["<div class='vd-empty-sm'><p>No active speaker profiles</p></div>"]

    total_dur = segments_list[-1]["end"] if segments_list else 0.0
    stats = f"""<div class="vd-stats">
  <span>⚡ {mkey} · {gpu_name} ({compute_dtype.upper()})</span>
  <span>{len(segments_list)} segments · {total_dur:.1f}s recorded ({elapsed_total:.1f}s compute)</span>
</div>"""

    full_transcript = "\n".join(html_parts) + stats

    # Export Files
    files = {}
    for ext, content in [
        ('.md',   f"# VoiceDiary Lecture Notes\n**Model:** {mkey} | **Compute:** {gpu_name}\n\n" + "\n\n".join(plain)),
        ('.txt',  "\n".join(plain)),
        ('.srt',  "\n".join(srt_parts)),
        ('.json', json.dumps(json_arr, indent=2, ensure_ascii=False))
    ]:
        tmp = tempfile.NamedTemporaryFile(mode='w', suffix=ext, delete=False, encoding='utf-8', prefix='VoiceDiary_')
        tmp.write(content); tmp.close()
        files[ext] = tmp.name

    export_html = f"""<div class="vd-export-row">
  <a class="vd-dl" href="/file={files['.md']}" download>
    <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/><line x1="12" y1="18" x2="12" y2="12"/><line x1="9" y1="15" x2="15" y2="15"/></svg>
    Markdown (.md)
  </a>
  <a class="vd-dl" href="/file={files['.txt']}" download>
    <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg>
    Plain Text (.txt)
  </a>
  <a class="vd-dl" href="/file={files['.srt']}" download>
    <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="2" y="2" width="20" height="20" rx="2"/><path d="M8 10h8M8 14h5"/></svg>
    Subtitles (.srt)
  </a>
  <a class="vd-dl" href="/file={files['.json']}" download>
    <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><polyline points="16 18 22 12 16 6"/><polyline points="8 6 2 12 8 18"/></svg>
    JSON Data (.json)
  </a>
</div>"""

    return full_transcript, "\n".join(sb), export_html, "\n".join(plain), files.get('.md'), files.get('.txt'), files.get('.srt')

# ─── Gemini 2.5 Flash Summarizer ───
def gemini_summary(text, key):
    if not text or not text.strip():
        return "⚠️ *Please transcribe a lecture first before generating an AI summary.*"
    api_key = (key or "").strip() or os.environ.get("GEMINI_API_KEY","")
    if not api_key:
        return "⚠️ *Please enter your Gemini API Key in the left sidebar to generate structured study notes.*"
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    prompt = (
        f"You are VoiceDiary AI, an elite university academic note-taker. "
        f"Analyze this multi-speaker classroom lecture transcript (Urdu and English code-switching) "
        f"and generate a comprehensive study guide formatted in clean Markdown:\n\n"
        f"## 📋 Executive Lecture Overview\n"
        f"(High-level synopsis of core themes)\n\n"
        f"## 🧠 Key Academic Concepts & Definitions\n"
        f"(Bulleted breakdown of technical points and explanations)\n\n"
        f"## 🎯 Critical Points for Exams\n"
        f"(Important concepts highlighted by the instructor)\n\n"
        f"## 💡 Notable Classroom Q&A & Discussion\n"
        f"(Summary of student questions and teacher answers)\n\n"
        f"Transcript:\n{text}"
    )
    payload = {"contents":[{"parts":[{"text": prompt}]}]}
    try:
        req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                     headers={"Content-Type":"application/json"})
        with urllib.request.urlopen(req, timeout=35) as r:
            res = json.loads(r.read())
            return res["candidates"][0]["content"]["parts"][0]["text"]
    except Exception as e:
        return f"❌ *Gemini AI Error: {e}*"

# ─── Modern High-Contrast CSS with Speaker Card Edit Button ───
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Fira+Code:wght@400;500;600&family=Noto+Nastaliq+Urdu:wght@400;700&display=swap');

html, body { margin: 0 !important; padding: 0 !important; height: 100% !important; background: #080C14 !important; }

.gradio-container {
  max-width: 100% !important;
  width: 100% !important;
  min-height: 100vh !important;
  margin: 0 !important;
  padding: 0 24px 32px 24px !important;
  background: #080C14 !important;
  font-family: 'Plus Jakarta Sans', -apple-system, BlinkMacSystemFont, sans-serif !important;
  color: #F8FAFC !important;
}

.gradio-container::before {
  content: '';
  position: fixed; inset: 0; z-index: 0; pointer-events: none;
  background:
    radial-gradient(ellipse 80% 45% at 50% -10%, rgba(99,102,241,.18) 0%, transparent 65%),
    radial-gradient(ellipse 60% 35% at 85% 90%, rgba(139,92,246,.10) 0%, transparent 55%);
}

/* Eliminate All Flickering */
.pending, .loading, .generating,
.gradio-container .pending,
.gradio-container [data-testid="loading"],
.gradio-container .loading {
  opacity: 1 !important;
  filter: none !important;
  transition: none !important;
  animation: none !important;
}
.loading-status, .progress-bar, .meta-text, .eta-bar,
.gradio-container .generating-status {
  display: none !important;
  visibility: hidden !important;
  opacity: 0 !important;
}

/* Audio Widget Clean Up */
.gradio-container .audio [data-testid="audio-time"],
.gradio-container .audio .timestamps,
.gradio-container .audio select,
.gradio-container .audio .source-select {
  display: none !important;
}
.gradio-container .audio button {
  display: inline-flex !important;
  visibility: visible !important;
  opacity: 1 !important;
}

.gradio-container .block,
.gradio-container .form,
.gradio-container .gap { box-shadow: none !important; border: none !important; background: transparent !important; }
.gr-group, .gr-box, .gr-panel { background: transparent !important; border: none !important; box-shadow: none !important; }
footer, .footer, .gr-footer { display: none !important; }

.gradio-container select,
.gradio-container input[type=text],
.gradio-container input[type=password],
.gradio-container textarea {
  background: rgba(15,23,42,.75) !important;
  border: 1px solid rgba(255,255,255,.12) !important;
  border-radius: 10px !important;
  color: #F8FAFC !important;
  font-family: inherit !important;
  font-size: 14px !important;
  font-weight: 500 !important;
  padding: 10px 14px !important;
}
.gradio-container select:focus,
.gradio-container input:focus {
  border-color: #6366F1 !important;
  box-shadow: 0 0 12px rgba(99,102,241,.35) !important;
  outline: none !important;
}

.gradio-container label > span,
.gradio-container .label-wrap span {
  color: #94A3B8 !important;
  font-size: 12px !important;
  font-weight: 700 !important;
  letter-spacing: .06em !important;
  text-transform: uppercase !important;
}

input[type=range] { accent-color: #6366F1 !important; }
.gradio-container .gr-slider input[type=number] {
  background: rgba(15,23,42,.9) !important;
  border: 1px solid rgba(255,255,255,.15) !important;
  color: #F8FAFC !important;
  border-radius: 8px !important;
  font-weight: 600 !important;
  font-size: 13px !important;
}

.tab-nav { background: transparent !important; border-bottom: 1px solid rgba(255,255,255,.10) !important; margin-bottom: 16px !important; }
.tab-nav button {
  color: #94A3B8 !important; font-weight: 700 !important; font-size: 14px !important;
  padding: 12px 24px !important; background: transparent !important;
  border: none !important; border-bottom: 2px solid transparent !important;
  border-radius: 0 !important; transition: all .15s !important;
}
.tab-nav button.selected { color: #FFFFFF !important; border-bottom-color: #6366F1 !important; background: rgba(99,102,241,.08) !important; }
.tabitem { background: transparent !important; border: none !important; padding: 4px 0 !important; }

.gradio-container .audio,
.gradio-container .gr-audio {
  background: rgba(15,23,42,.65) !important;
  border: 1px solid rgba(255,255,255,.10) !important;
  border-radius: 14px !important;
  overflow: visible !important;
  min-height: 85px !important;
  padding: 12px !important;
}

.gradio-container button.primary,
.vd-btn-primary {
  background: linear-gradient(135deg,#6366F1 0%,#8B5CF6 100%) !important;
  color: #FFFFFF !important; border: none !important;
  font-weight: 700 !important; font-size: 15px !important;
  border-radius: 12px !important; padding: 14px 28px !important;
  box-shadow: 0 0 24px rgba(99,102,241,.35) !important;
  transition: all .2s !important; width: 100% !important; cursor: pointer !important;
}
.gradio-container button.primary:hover,
.vd-btn-primary:hover {
  transform: translateY(-2px) !important;
  box-shadow: 0 0 32px rgba(99,102,241,.55) !important;
}

.gradio-container button.secondary {
  background: rgba(255,255,255,.06) !important;
  border: 1px solid rgba(255,255,255,.12) !important;
  color: #E2E8F0 !important; font-weight: 600 !important; font-size: 13px !important;
  border-radius: 10px !important; padding: 10px 18px !important; transition: all .15s !important;
}
.gradio-container button.secondary:hover { background: rgba(255,255,255,.12) !important; color: #FFFFFF !important; }

.vd-hdr {
  display: flex; align-items: center; justify-content: space-between;
  padding: 18px 0; margin-bottom: 24px;
  border-bottom: 1px solid rgba(255,255,255,.08);
}
.vd-hdr-left { display: flex; align-items: center; gap: 14px; }
.vd-logo-box {
  width: 44px; height: 44px; border-radius: 12px; flex-shrink: 0;
  background: linear-gradient(135deg,#6366F1,#8B5CF6);
  display: flex; align-items: center; justify-content: center;
  box-shadow: 0 0 20px rgba(99,102,241,.45);
}
.vd-hdr-title { font-size: 22px; font-weight: 800; color: #FFFFFF; letter-spacing: -.02em; }
.vd-hdr-sub { font-size: 13px; color: #94A3B8; font-weight: 500; margin-top: 2px; }
.vd-hw-pill {
  display: inline-flex; align-items: center; gap: 8px;
  padding: 7px 18px; border-radius: 9999px;
  background: rgba(16,185,129,.12); border: 1px solid rgba(16,185,129,.28);
  font-size: 12px; font-weight: 700; color: #10B981;
  font-family: 'Fira Code', monospace;
}
.vd-hw-dot { width: 8px; height: 8px; border-radius: 50%; background: #10B981; box-shadow: 0 0 10px rgba(16,185,129,.8); }

.vd-sec-lbl {
  font-size: 12px; font-weight: 800; color: #94A3B8;
  letter-spacing: .08em; text-transform: uppercase;
  display: flex; align-items: center; justify-content: space-between;
  margin-bottom: 12px; margin-top: 18px;
}
.vd-sec-lbl:first-child { margin-top: 0; }
.vd-badge-live {
  width: 8px; height: 8px; border-radius: 50%; background: #10B981;
  box-shadow: 0 0 10px rgba(16,185,129,.9); display: inline-block; margin-right: 6px;
  animation: pulse 2s infinite;
}
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:.4} }

.vd-spk-card {
  display: flex; align-items: center; gap: 12px;
  padding: 12px 14px; border-radius: 14px;
  background: rgba(15,23,42,.65); border: 1px solid rgba(255,255,255,.08);
  margin-bottom: 8px; transition: .15s ease;
}
.vd-spk-card:hover { background: rgba(30,41,59,.75); border-color: rgba(255,255,255,.16); transform: translateX(2px); }
.vd-spk-av {
  width: 38px; height: 38px; border-radius: 50%; flex-shrink: 0;
  display: flex; align-items: center; justify-content: center;
  font-weight: 800; font-size: 14px; color: #FFFFFF;
}
.vd-spk-info { flex: 1; min-width: 0; }
.vd-spk-name { font-size: 14px; font-weight: 700; color: #F1F5F9; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.vd-spk-meta { font-size: 11px; color: #64748B; margin-top: 2px; }

/* Inline Edit Button on Speaker Card */
.vd-spk-edit-btn {
  display: inline-flex !important;
  align-items: center !important;
  gap: 5px !important;
  padding: 6px 12px !important;
  border-radius: 8px !important;
  background: rgba(255,255,255,.06) !important;
  border: 1px solid rgba(255,255,255,.14) !important;
  color: #94A3B8 !important;
  font-size: 12px !important;
  font-weight: 600 !important;
  cursor: pointer !important;
  transition: all .15s ease !important;
  flex-shrink: 0 !important;
}
.vd-spk-edit-btn:hover {
  background: rgba(99,102,241,.25) !important;
  border-color: rgba(99,102,241,.6) !important;
  color: #FFFFFF !important;
  transform: scale(1.04) !important;
}

.vd-transcript-vp {
  background: rgba(10,15,28,.85); border: 1px solid rgba(255,255,255,.08);
  border-radius: 16px; padding: 22px; min-height: 380px; max-height: 520px;
  overflow-y: auto; backdrop-filter: blur(16px);
}
.vd-seg {
  margin-bottom: 14px; border-radius: 12px;
  background: rgba(15,23,42,.75); border: 1px solid rgba(255,255,255,.08);
  border-left-width: 5px; border-left-style: solid;
  overflow: hidden; transition: .15s ease;
}
.vd-seg:hover { background: rgba(30,41,59,.85); border-color: rgba(255,255,255,.16); }
.vd-seg-body { padding: 16px 20px; }
.vd-seg-meta { display: flex; align-items: center; gap: 10px; margin-bottom: 8px; }
.vd-dot { width: 8px; height: 8px; border-radius: 50%; display: inline-block; }
.vd-spk-lbl { font-size: 14px; font-weight: 700; }
.vd-time { font-size: 12px; color: #64748B; font-family: 'Fira Code', monospace; }
.vd-txt { font-size: 16px; line-height: 1.7; color: #F1F5F9; word-break: break-word; font-weight: 400; }
.vd-rtl { direction: rtl; text-align: right; font-family: 'Noto Nastaliq Urdu', serif; font-size: 20px; line-height: 2.2; color: #FFFFFF; }

.vd-stats {
  margin-top: 16px; padding-top: 14px; border-top: 1px solid rgba(255,255,255,.08);
  display: flex; justify-content: space-between; align-items: center;
  font-size: 12px; color: #64748B; font-family: 'Fira Code', monospace;
}

.vd-empty {
  display: flex; flex-direction: column; align-items: center; justify-content: center;
  padding: 80px 24px; text-align: center; color: #475569; gap: 12px;
}
.vd-empty p { font-size: 17px; font-weight: 700; color: #94A3B8; margin: 0; }
.vd-empty span { font-size: 13px; color: #64748B; }
.vd-empty-sm {
  display: flex; flex-direction: column; align-items: center;
  padding: 24px 12px; text-align: center; color: #475569; gap: 8px;
}
.vd-empty-sm p { font-size: 13px; font-weight: 600; color: #94A3B8; margin: 0; }
.vd-empty-sm span { font-size: 11px; color: #64748B; }

.vd-export-row { display: flex; gap: 12px; flex-wrap: wrap; margin-top: 6px; }
.vd-dl {
  display: inline-flex; align-items: center; gap: 8px;
  padding: 11px 18px; border-radius: 10px;
  background: rgba(255,255,255,.05); border: 1px solid rgba(255,255,255,.12);
  color: #E2E8F0; font-size: 14px; font-weight: 600;
  text-decoration: none; transition: .15s ease; cursor: pointer;
}
.vd-dl:hover { background: rgba(99,102,241,.18); border-color: rgba(99,102,241,.45); color: #A5B4FC; transform: translateY(-1px); }
.vd-dl.disabled { opacity: 0.4; cursor: not-allowed; pointer-events: none; }
.vd-divider { border: none; border-top: 1px solid rgba(255,255,255,.08); margin: 16px 0; }

.vd-summary-card {
  background: rgba(15,23,42,.75); border: 1px solid rgba(255,255,255,.10);
  border-radius: 16px; padding: 24px; font-size: 15px; line-height: 1.7;
  color: #F1F5F9; backdrop-filter: blur(16px);
}
"""

HEAD_JS = """
<script>
window.renameSpeaker = function(spkId, oldName) {
  const newName = prompt(`Enter new display name for Speaker ${spkId}:`, oldName);
  if (newName !== null && newName.trim() !== "" && newName.trim() !== oldName) {
    const input = document.querySelector('#hidden_rename_payload input, #hidden_rename_payload textarea');
    const btn = document.querySelector('#hidden_rename_trigger');
    if (input && btn) {
      input.value = JSON.stringify({id: spkId, name: newName.trim()});
      input.dispatchEvent(new Event('input', { bubbles: true }));
      btn.click();
    }
  }
};
</script>
"""

with gr.Blocks(title="VoiceDiary — AI Bilingual Lecture & Diarization Engine", css=CSS, head=HEAD_JS,
               theme=gr.themes.Default(primary_hue="indigo", neutral_hue="slate")) as demo:

    # Global Session State
    transcript_state = gr.State("")
    session_segments_state = gr.State([])      # List of [{speaker, id, start, end, time, text}]
    speaker_profiles_state = gr.State({})      # Dict of {speaker_id: count}
    speaker_names_state = gr.State({})         # Dict of {speaker_id: custom_name}
    speaker_embeddings_state = gr.State({})    # Dict of {speaker_id: [emb_vectors]}
    live_audio_buffer = gr.State(np.array([], dtype=np.float32))
    last_processed_sec = gr.State(0.0)
    total_compute_time = gr.State(0.0)

    # ── HEADER ──
    gr.HTML(f"""
    <div class="vd-hdr">
      <div class="vd-hdr-left">
        <div class="vd-logo-box">
          <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2.2">
            <path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/>
            <path d="M19 10v2a7 7 0 0 1-14 0v-2"/>
            <line x1="12" y1="19" x2="12" y2="23"/><line x1="8" y1="23" x2="16" y2="23"/>
          </svg>
        </div>
        <div>
          <div class="vd-hdr-title">VoiceDiary</div>
          <div class="vd-hdr-sub">Bilingual Classroom Lecture Note-Taking &amp; Neural Diarization Engine</div>
        </div>
      </div>
      <div class="vd-hw-pill">
        <span class="vd-hw-dot"></span>
        {gpu_name} &nbsp;·&nbsp; Tensor Cores {compute_dtype.upper()}
      </div>
    </div>""")

    with gr.Row(equal_height=False):
        # ── LEFT SIDEBAR (Clean, Dynamic Cards with Inline Edit) ──
        with gr.Column(scale=3, min_width=280):
            gr.HTML("<div class='vd-sec-lbl'><span>Speakers &amp; Neural Profiles</span><span><span class='vd-badge-live'></span>LIVE</span></div>")
            sidebar_out = gr.HTML(value="""<div class='vd-empty-sm'>
              <svg width='36' height='36' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
                <path d='M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2'/><circle cx='9' cy='7' r='4'/>
                <path d='M23 21v-2a4 4 0 0 0-3-3.87'/><path d='M16 3.13a4 4 0 0 1 0 7.75'/>
              </svg>
              <p>No active speaker profiles</p>
              <span>Start recording to build neural voiceprints</span>
            </div>""")

            # Hidden bridge for inline JS speaker rename
            with gr.Row(visible=False):
                hidden_rename_payload = gr.Textbox(elem_id="hidden_rename_payload")
                hidden_rename_trigger = gr.Button("Trigger Rename", elem_id="hidden_rename_trigger")

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>AI Engine &amp; Whisper Model</span></div>")
            model_dd = gr.Dropdown(choices=list(MODEL_MAP.keys()),
                value='Large-v3-Turbo (809M)', label='Active Whisper Model')
            lang_dd = gr.Dropdown(choices=list(LANG_MAP.keys()),
                value='Bilingual (Urdu + English)', label='Language Output Mode')

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Diarization Sensitivity</span></div>")
            thresh_sl = gr.Slider(20, 70, 32, step=1, label='Cosine Similarity Threshold (%)')
            vad_sl = gr.Slider(150, 600, 280, step=10, label='VAD Silence Gap (ms)')

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Google Gemini AI (BYOK)</span></div>")
            gemini_key = gr.Textbox(placeholder='Paste Gemini API Key (AIzaSy…)', type='password',
                                    label='Gemini API Key', container=False)

        # ── RIGHT MAIN WORKSPACE ──
        with gr.Column(scale=9, min_width=520):
            with gr.Tabs():
                with gr.TabItem("🎙️ Live Classroom Microphone"):
                    audio_mic = gr.Audio(sources=["microphone"], type="numpy", streaming=True,
                                         label="Microphone", show_label=False)
                    clear_live_btn = gr.Button("Clear Live Session", variant="secondary", size="sm")
                with gr.TabItem("📁 Upload Pre-Recorded Lecture"):
                    audio_file = gr.Audio(sources=["upload"], type="filepath",
                                          label="Upload classroom audio (.wav, .mp3, .m4a, .flac)",
                                          show_label=False)
                    transcribe_file_btn = gr.Button("⚡ Transcribe & Diarize Uploaded Audio (GPU)",
                                                    variant="primary", elem_classes=["vd-btn-primary"])

            gr.HTML("<div class='vd-sec-lbl' style='margin-top:22px;'><span>Classroom Lecture Transcript</span></div>")
            transcript_out = gr.HTML(
                value="""<div class='vd-empty'>
                  <svg width='54' height='54' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
                    <path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/>
                  </svg>
                  <p>Lecture transcript will stream live here</p>
                  <span>Speak into the microphone or upload audio</span>
                </div>""",
                elem_classes=["vd-transcript-vp"])

            gr.HTML("<div class='vd-sec-lbl' style='margin-top:20px;'><span>Export Lecture Notes</span></div>")
            export_html_out = gr.HTML(value="""<div class='vd-export-row'>
              <span class='vd-dl disabled'>Markdown (.md)</span>
              <span class='vd-dl disabled'>Plain Text (.txt)</span>
              <span class='vd-dl disabled'>Subtitles (.srt)</span>
              <span class='vd-dl disabled'>JSON (.json)</span>
            </div>""")

            with gr.Row(visible=False):
                f_md  = gr.File()
                f_txt = gr.File()
                f_srt = gr.File()

            gr.HTML("<hr class='vd-divider' style='margin:26px 0 20px 0;'>")
            gr.HTML("<div class='vd-sec-lbl'><span>AI Study Summary &amp; Flashcards (Gemini 2.5 Flash)</span></div>")
            ai_btn = gr.Button("✨ Generate AI Lecture Summary & Study Guide",
                               variant="primary", elem_classes=["vd-btn-primary"])
            ai_out = gr.Markdown(value="*AI study summary and key concepts will be generated here after clicking above.*",
                                 elem_classes=["vd-summary-card"])

    # ── EVENT WIRINGS ──

    # Incremental 5-second streaming chunk processor (Creates a NEW BOX per segment!)
    def handle_audio_stream(chunk, buffer, last_sec, segments_list, profiles_dict, names_dict, embeddings_dict, compute_time,
                            model_choice, lang_choice, thresh_pct, vad_ms):
        if chunk is None:
            return buffer, last_sec, segments_list, profiles_dict, names_dict, embeddings_dict, compute_time, gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip()

        y = convert_chunk_to_16k(chunk)
        if y is None or len(y) == 0:
            return buffer, last_sec, segments_list, profiles_dict, names_dict, embeddings_dict, compute_time, gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip()

        if buffer is None or len(buffer) == 0:
            buffer = y
        else:
            buffer = np.append(buffer, y)

        current_dur = len(buffer) / 16000.0
        # Process every ~4.5 seconds of new speech
        if (current_dur - last_sec) >= 4.5 or (last_sec == 0.0 and current_dur >= 2.0):
            t0 = time.time()
            start_sample = int(last_sec * 16000)
            chunk_slice = buffer[start_sample:]
            chunk_start_sec = last_sec
            chunk_end_sec = current_dur
            last_sec = current_dur

            # Noise floor check
            rms = float(np.sqrt(np.mean(chunk_slice ** 2)))
            if rms >= 0.003 and len(chunk_slice) >= 4000:
                mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
                model = get_model(mkey)
                target_lang, is_urdu_script, is_english_only = LANG_MAP.get(lang_choice, ('ur', False, False))
                is_roman = (lang_choice == 'Roman Urdu (Latin)')
                thresh = float(thresh_pct) / 100.0

                try:
                    segs, info = model.transcribe(
                        chunk_slice,
                        language=target_lang,
                        beam_size=1, best_of=1, temperature=0.0,
                        initial_prompt=PAKISTANI_LECTURE_PROMPT,
                        condition_on_previous_text=False,
                        without_timestamps=False,
                        no_speech_threshold=0.6,
                        compression_ratio_threshold=2.4,
                        vad_filter=True,
                        vad_parameters=dict(min_silence_duration_ms=int(vad_ms))
                    )
                    
                    # Diarize chunk
                    spk = 1
                    if len(chunk_slice) >= 8000:
                        try:
                            with torch.inference_mode():
                                w = torch.from_numpy(chunk_slice).float().unsqueeze(0).to(device_type)
                                e = embedder.encode_batch(w).squeeze().detach().cpu().numpy()
                                en = e / (np.linalg.norm(e) or 1.)
                            bid, bsim = None, -1.
                            for sid, embs in embeddings_dict.items():
                                ms_ = max(float(np.dot(en, x)) for x in embs) if embs else 0
                                if ms_ > bsim: bsim, bid = ms_, sid
                            if bid and bsim >= thresh:
                                spk = bid
                                if len(embeddings_dict[spk]) < 50: embeddings_dict[spk].append(en)
                            else:
                                spk = len(embeddings_dict) + 1
                                embeddings_dict[spk] = [en]
                            profiles_dict[spk] = len(embeddings_dict[spk])
                        except Exception: pass

                    # Default name if not yet named
                    if spk not in names_dict:
                        names_dict[spk] = f"Speaker {spk}"

                    for s in segs:
                        raw = s.text.strip()
                        if not raw: continue
                        conf = float(getattr(s, "avg_logprob", getattr(s, "avg_log_prob", 0.0)))
                        no_speech_prob = float(getattr(s, "no_speech_prob", 0.0))
                        if no_speech_prob > 0.65 or conf < -1.2: continue
                        raw = re.sub(r'\b(\w+(?:\s+\w+){0,3})(?:\s+\1\b)+', r'\1', raw, flags=re.IGNORECASE).strip()
                        raw = sanitize_script(raw)
                        if not raw or len(raw) < 2: continue

                        if is_urdu_script:
                            txt_clean = UrduNormalizer.normalize(raw)
                        elif is_english_only:
                            txt_clean = post_process_english(raw)
                        elif is_roman:
                            norm_ur = UrduNormalizer.normalize(raw)
                            txt_clean = post_process_english(to_roman_urdu(norm_ur))
                        else:
                            words = raw.split()
                            proc = []
                            for w in words:
                                if any('\u0600' <= ch <= '\u06FF' for ch in w):
                                    proc.append(UrduNormalizer.normalize(w))
                                else:
                                    proc.append(w)
                            txt_clean = post_process_english(" ".join(proc))

                        seg_start = chunk_start_sec + s.start
                        seg_end = chunk_start_sec + s.end
                        ts = f"{int(seg_start//60):02d}:{int(seg_start%60):02d}"
                        
                        segments_list.append({
                            "speaker": names_dict.get(spk, f"Speaker {spk}"),
                            "id": spk,
                            "start": seg_start,
                            "end": seg_end,
                            "time": ts,
                            "text": txt_clean
                        })
                except Exception as e:
                    print("Stream chunk error:", e)

            compute_time += (time.time() - t0)
            mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
            t_html, s_html, exp_html, plain_txt, f1, f2, f3 = render_transcript_ui(
                segments_list, profiles_dict, names_dict, mkey, compute_time
            )
            return buffer, last_sec, segments_list, profiles_dict, names_dict, embeddings_dict, compute_time, t_html, s_html, exp_html, plain_txt, f1, f2, f3

        return buffer, last_sec, segments_list, profiles_dict, names_dict, embeddings_dict, compute_time, gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip(), gr.skip()

    audio_mic.stream(
        fn=handle_audio_stream,
        inputs=[audio_mic, live_audio_buffer, last_processed_sec, session_segments_state, speaker_profiles_state, speaker_names_state, speaker_embeddings_state, total_compute_time,
                model_dd, lang_dd, thresh_sl, vad_sl],
        outputs=[live_audio_buffer, last_processed_sec, session_segments_state, speaker_profiles_state, speaker_names_state, speaker_embeddings_state, total_compute_time,
                 transcript_out, sidebar_out, export_html_out, transcript_state, f_md, f_txt, f_srt],
        show_progress="hidden"
    )

    # Clear Live Session
    def reset_live_session(model_choice):
        mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
        empty_t, empty_s, empty_e, _, _, _, _ = render_transcript_ui([], {}, {}, mkey, 0.0)
        return (
            np.array([], dtype=np.float32), 0.0, [], {}, {}, {}, 0.0,
            empty_t, empty_s, empty_e, "", None, None, None
        )
    
    clear_live_btn.click(
        fn=reset_live_session,
        inputs=[model_dd],
        outputs=[live_audio_buffer, last_processed_sec, session_segments_state, speaker_profiles_state, speaker_names_state, speaker_embeddings_state, total_compute_time,
                 transcript_out, sidebar_out, export_html_out, transcript_state, f_md, f_txt, f_srt],
        show_progress="hidden"
    )

    # Inline Rename Handler triggered from Speaker Card Edit button
    def handle_inline_rename(payload_json, segments_list, profiles_dict, names_dict, compute_time, model_choice):
        if payload_json:
            try:
                data = json.loads(payload_json)
                sid = int(data.get("id", 1))
                new_name = str(data.get("name", "")).strip()
                if new_name:
                    names_dict[sid] = new_name
                    for seg in segments_list:
                        if seg["id"] == sid:
                            seg["speaker"] = new_name
            except Exception as e:
                print("Rename error:", e)

        mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
        t_html, s_html, exp_html, plain_txt, f1, f2, f3 = render_transcript_ui(
            segments_list, profiles_dict, names_dict, mkey, compute_time
        )
        return names_dict, segments_list, t_html, s_html, exp_html, plain_txt, f1, f2, f3

    hidden_rename_trigger.click(
        fn=handle_inline_rename,
        inputs=[hidden_rename_payload, session_segments_state, speaker_profiles_state, speaker_names_state, total_compute_time, model_dd],
        outputs=[speaker_names_state, session_segments_state, transcript_out, sidebar_out, export_html_out, transcript_state, f_md, f_txt, f_srt]
    )

    # Upload Pre-recorded Audio Batch Transcribe
    def handle_file_upload(fpath, model_choice, lang_choice, thresh_pct, vad_ms):
        mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
        if not fpath:
            return {}, [], render_transcript_ui([], {}, {}, mkey, 0.0)
        
        t0 = time.time()
        data = load_16k_from_file(fpath)
        if len(data) < 4000:
            return {}, [], render_transcript_ui([], {}, {}, mkey, 0.0)

        model = get_model(mkey)
        target_lang, is_urdu_script, is_english_only = LANG_MAP.get(lang_choice, ('ur', False, False))
        is_roman = (lang_choice == 'Roman Urdu (Latin)')
        thresh = float(thresh_pct) / 100.0

        try:
            segs, info = model.transcribe(
                data,
                language=target_lang,
                beam_size=1, best_of=1, temperature=0.0,
                initial_prompt=PAKISTANI_LECTURE_PROMPT,
                condition_on_previous_text=False,
                without_timestamps=False,
                no_speech_threshold=0.6,
                compression_ratio_threshold=2.4,
                vad_filter=True,
                vad_parameters=dict(min_silence_duration_ms=int(vad_ms))
            )
        except Exception as e:
            return {}, [], f"<div class='vd-empty'><p>Transcription error: {e}</p></div>", "", "", "", None, None, None

        segments_list = []
        profiles = {}
        names = {}
        embeddings = {}
        
        for s in segs:
            raw = s.text.strip()
            if not raw: continue
            conf = float(getattr(s, "avg_logprob", getattr(s, "avg_log_prob", 0.0)))
            no_speech_prob = float(getattr(s, "no_speech_prob", 0.0))
            if no_speech_prob > 0.65 or conf < -1.2: continue
            raw = re.sub(r'\b(\w+(?:\s+\w+){0,3})(?:\s+\1\b)+', r'\1', raw, flags=re.IGNORECASE).strip()
            raw = sanitize_script(raw)
            if not raw or len(raw) < 2: continue

            if is_urdu_script:
                txt_clean = UrduNormalizer.normalize(raw)
            elif is_english_only:
                txt_clean = post_process_english(raw)
            elif is_roman:
                norm_ur = UrduNormalizer.normalize(raw)
                txt_clean = post_process_english(to_roman_urdu(norm_ur))
            else:
                words = raw.split()
                proc = []
                for w in words:
                    if any('\u0600' <= ch <= '\u06FF' for ch in w):
                        proc.append(UrduNormalizer.normalize(w))
                    else:
                        proc.append(w)
                txt_clean = post_process_english(" ".join(proc))

            s0, s1 = s.start, s.end
            chunk = data[int(s0*16000):int(s1*16000)]
            spk = 1
            if len(chunk) >= 8000:
                try:
                    with torch.inference_mode():
                        w = torch.from_numpy(chunk).float().unsqueeze(0).to(device_type)
                        e = embedder.encode_batch(w).squeeze().detach().cpu().numpy()
                        en = e / (np.linalg.norm(e) or 1.)
                    bid, bsim = None, -1.
                    for sid, embs in embeddings.items():
                        ms_ = max(float(np.dot(en, x)) for x in embs) if embs else 0
                        if ms_ > bsim: bsim, bid = ms_, sid
                    if bid and bsim >= thresh:
                        spk = bid
                        if len(embeddings[spk]) < 50: embeddings[spk].append(en)
                    else:
                        spk = len(embeddings) + 1
                        embeddings[spk] = [en]
                    profiles[spk] = len(embeddings[spk])
                except: pass

            if spk not in names:
                names[spk] = f"Speaker {spk}"

            ts = f"{int(s0//60):02d}:{int(s0%60):02d}"
            segments_list.append({
                "speaker": names[spk],
                "id": spk,
                "start": s0,
                "end": s1,
                "time": ts,
                "text": txt_clean
            })

        elapsed = time.time() - t0
        t_html, s_html, exp_html, plain_txt, f1, f2, f3 = render_transcript_ui(segments_list, profiles, names, mkey, elapsed)
        return names, segments_list, profiles, t_html, s_html, exp_html, plain_txt, f1, f2, f3

    transcribe_file_btn.click(
        fn=handle_file_upload,
        inputs=[audio_file, model_dd, lang_dd, thresh_sl, vad_sl],
        outputs=[speaker_names_state, session_segments_state, speaker_profiles_state, transcript_out, sidebar_out, export_html_out, transcript_state, f_md, f_txt, f_srt]
    )

    ai_btn.click(fn=gemini_summary, inputs=[transcript_state, gemini_key], outputs=[ai_out])

demo.queue(max_size=20).launch(share=True, inline=True, debug=False, show_error=True)
